# AI Agent 실습: 업무 요청 자동 분류 & 처리 비서

지금까지 배운 `PromptTemplate`, `ChatPromptTemplate`, `chain = prompt | llm`, `StrOutputParser`를 확장해서,
**"입력 → 의도 분류 → 적절한 프롬프트 선택 → 답변 생성"** 구조를 만들어봅니다.

이 구조는 간단한 AI Agent의 사고 흐름(판단 → 도구/프롬프트 선택 → 실행)과 동일합니다.

- 새로 배우는 기능: `RunnableLambda`, `RunnableBranch`


## STEP 1. 기본 설정

`.env`에서 API 키를 불러오고 LLM을 준비합니다. `temperature=0`으로 설정한 이유는, 뒤에서 만들 "의도 분류기"가 매번 같은 입력에 대해 같은 카테고리를 안정적으로 뽑아내야 하기 때문입니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatOpenAI(model_name="gpt-4o", temperature=0)


## STEP 2. "의도 분류" 전용 체인 만들기

Agent가 "무엇을 해야 할지" 판단하는 단계입니다. 별도의 분류 모델 없이, LLM에게 "카테고리 이름 한 단어만 출력해"라고 프롬프트로 지시해서 분류기처럼 사용합니다.


In [2]:
intent_prompt = PromptTemplate.from_template(
    """다음 사용자 요청을 아래 카테고리 중 하나로만 분류하세요.
반드시 카테고리 이름 한 단어만 출력하고, 다른 설명은 붙이지 마세요.

[카테고리]
- EMAIL: 이메일/메시지 작성을 요청하는 경우
- SUMMARY: 글이나 문서 요약을 요청하는 경우
- CODE: 코드 설명, 디버깅, 코드 작성을 요청하는 경우
- TRANSLATE: 번역을 요청하는 경우
- GENERAL: 위에 해당하지 않는 일반 대화/질문

[사용자 요청]
{user_input}

[카테고리]"""
)

intent_chain = intent_prompt | llm | StrOutputParser()

# 테스트
intent_chain.invoke({"user_input": "이 파이썬 코드가 뭘 하는건지 설명해줘"})


'CODE'

## STEP 3. 카테고리별 전용 프롬프트 + 체인 만들기

카테고리마다 말투/형식/역할이 다르므로 프롬프트를 따로 둡니다. 각각을 `prompt | llm | StrOutputParser()`로 완성된 체인으로 만들어두면, 나중에 "이 요청이 EMAIL이면 `chain_email`을 실행"이라고 그대로 가져다 쓸 수 있습니다.


In [4]:
email_prompt = PromptTemplate.from_template(
    "다음 요청에 맞는 정중한 비즈니스 이메일을 작성해줘. 요청: {user_input}"
)
summary_prompt = PromptTemplate.from_template(
    "다음 내용을 3줄로 핵심만 요약해줘. 내용: {user_input}"
)
code_prompt = PromptTemplate.from_template(
    "다음 코드/요청에 대해 초보자도 이해하기 쉽게 설명해줘. {user_input}"
)
translate_prompt = PromptTemplate.from_template(
    "다음 문장을 자연스러운 영어로 번역해줘. 문장: {user_input}"
)
general_prompt = PromptTemplate.from_template("{user_input}")

chain_email = email_prompt | llm | StrOutputParser()
chain_summary = summary_prompt | llm | StrOutputParser()
chain_code = code_prompt | llm | StrOutputParser()
chain_translate = translate_prompt | llm | StrOutputParser()
chain_general = general_prompt | llm | StrOutputParser()


## STEP 4. `RunnableLambda`로 "분류 결과 붙이기" 단계 만들기

`RunnableLambda`는 일반 파이썬 함수를 체인(`|`)에 끼워 넣을 수 있게 해주는 도구입니다. 여기서는 "분류 체인을 호출해서 그 결과를 딕셔너리에 추가하는" 커스텀 로직이 필요한데, 이런 자유로운 파이썬 로직을 파이프라인 중간에 넣고 싶을 때 사용합니다.


In [5]:
from langchain_core.runnables import RunnableLambda

def add_intent(input_dict: dict) -> dict:
    label = intent_chain.invoke({"user_input": input_dict["user_input"]})
    label = label.strip().upper()
    return {"user_input": input_dict["user_input"], "intent": label}

add_intent_step = RunnableLambda(add_intent)

# 테스트
add_intent_step.invoke({"user_input": "김대리한테 회의 일정 변경 메일 좀 써줘"})


{'user_input': '김대리한테 회의 일정 변경 메일 좀 써줘', 'intent': 'EMAIL'}

## STEP 5. `RunnableBranch`로 "의도에 따라 다른 체인 실행" 만들기

`RunnableBranch`는 `(조건함수, 실행할체인)` 쌍을 여러 개 등록해두고, 입력이 들어오면 위에서부터 조건을 순서대로 검사해서 **처음 맞는 조건의 체인**을 실행하는 라우터입니다. 마지막에 조건 없이 넣은 체인은 "이 중 아무것도 아니면 실행할 기본값(default)"이 됩니다. if-elif-else를 체인 버전으로 쓰는 것이라고 생각하면 됩니다.


In [6]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: x["intent"] == "EMAIL", chain_email),
    (lambda x: x["intent"] == "SUMMARY", chain_summary),
    (lambda x: x["intent"] == "CODE", chain_code),
    (lambda x: x["intent"] == "TRANSLATE", chain_translate),
    chain_general,  # 위 조건에 하나도 안 맞으면 이게 실행됨 (default)
)


## STEP 6. 전체 파이프라인 결합 & 테스트

`add_intent_step | branch`가 바로 "입력 → 의도 분류 → 프롬프트 선택 → 답변 생성"이라는 전체 사고 흐름을 하나의 체인으로 묶은 것입니다. `full_chain.invoke({"user_input": ...})` 한 줄만 호출하면 내부적으로 분류와 라우팅이 자동으로 일어납니다.


In [8]:
full_chain = add_intent_step | branch

# 여러 입력으로 테스트
tests = [
    "김대리한테 회의 일정 변경 메일 좀 써줘",
    "다음 보고서 내용을 요약해줘: (긴 텍스트...)",
    "이 코드가 왜 에러가 나는지 설명해줘: print(1/0)",
    "이 문장을 영어로 번역해줘: 오늘 날씨가 좋네요",
    "오늘 점심 뭐 먹을지 추천해줘",
]

for t in tests:
    print(f"[입력] {t}")
    print(f"[답변] {full_chain.invoke({'user_input': t})}")


[입력] 김대리한테 회의 일정 변경 메일 좀 써줘
[답변] 제목: 회의 일정 변경 안내

김대리님께,

안녕하세요, 김대리님. 잘 지내고 계신가요?

다름이 아니라, 예정되어 있던 회의 일정에 변경이 있어 안내드리고자 합니다. 원래 예정되었던 회의는 [기존 날짜와 시간]에 진행될 예정이었으나, 내부 사정으로 인해 [새로운 날짜와 시간]로 변경하게 되었습니다.

회의 장소는 변동 없이 [회의 장소]에서 진행될 예정입니다. 변경된 일정이 김대리님의 업무에 지장을 주지 않기를 바라며, 참석 가능 여부를 알려주시면 감사하겠습니다.

혹시 새로운 일정에 어려움이 있으시다면 언제든지 말씀해 주세요. 최대한 조율할 수 있도록 하겠습니다.

감사합니다.

좋은 하루 보내세요.

[당신의 이름]
[당신의 직책]
[회사 이름]
[연락처 정보]
[입력] 다음 보고서 내용을 요약해줘: (긴 텍스트...)
[답변] 죄송하지만, 제공된 텍스트가 없어 요약을 할 수 없습니다. 요약을 원하시는 텍스트를 입력해 주시면 도와드리겠습니다.
[입력] 이 코드가 왜 에러가 나는지 설명해줘: print(1/0)
[답변] 물론입니다! 이 코드는 Python 프로그래밍 언어로 작성된 간단한 코드입니다. `print(1/0)`는 숫자 1을 숫자 0으로 나눈 결과를 출력하려고 합니다.

이 코드가 에러가 나는 이유는 수학적으로 0으로 나누는 것이 정의되지 않기 때문입니다. 즉, 어떤 숫자도 0으로 나눌 수 없습니다. 예를 들어, 10개의 사과를 0명의 사람에게 나누어 주는 상황을 상상해보세요. 이는 불가능한 일이죠. 그래서 컴퓨터도 이 상황을 처리할 수 없고, 에러를 발생시킵니다.

Python에서는 0으로 나누려고 할 때 `ZeroDivisionError`라는 에러가 발생합니다. 이 에러는 "0으로 나눌 수 없다"는 것을 알려주는 메시지입니다. 따라서 이 코드를 실행하면 프로그램이 멈추고 에러 메시지를 출력하게 됩니다.

이 문제를 해결하려면 0이 아닌 다른 숫자로 나누거나, 나누기 전에 나누는 수가 0

## 연습문제

1. **카테고리 추가하기**: `COMPLAINT`(불만/컴플레인 대응) 카테고리를 하나 추가해보세요. `intent_prompt`의 카테고리 목록에 설명을 추가하고, 전용 `complaint_prompt`/`chain_complaint`를 만든 뒤 `branch`에 조건을 하나 더 넣으면 됩니다.

2. **분류 신뢰도 처리**: `intent_chain`이 예상한 5개 카테고리 이외의 이상한 값(오타, 여러 단어 등)을 출력하면 어떻게 될까요? `add_intent` 함수 안에서 `label`이 `["EMAIL","SUMMARY","CODE","TRANSLATE","GENERAL"]`에 없을 경우 강제로 `"GENERAL"`로 바꾸는 방어 코드를 추가해보세요.
